# ***定位部分主要逻辑***

# 1. 坐标转换（图像 → 地图）

## 数学原理

使用**透视变换**将图像平面上的点映射到地图平面上的点。  
齐次坐标表示为：

$$
\begin{bmatrix} x' \\ y' \\ w' \end{bmatrix} = \mathbf{M} \cdot \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

$$
X_{\text{map}} = \frac{x'}{w'}, \quad Y_{\text{map}} = \frac{y'}{w'}
$$

由于战场存在不同高度的地形（地面、R型高地、环形高地），每个高度层使用不同的变换矩阵 $\mathbf{M}_{\text{ground}}$、$\mathbf{M}_{\text{heightR}}$、$\mathbf{M}_{\text{heightG}}$。

**分层判断流程**：
1. 先用 $\mathbf{M}_{\text{ground}}$ 初步变换，得到中间坐标 $(x_t, y_t)$。
2. 根据掩码图像在该坐标处的颜色确定真实高度层：
   - 黑色 → 地面层
   - 绿色分量最大 → R型高地
   - 蓝色分量最大 → 环形高地
3. 使用对应层的矩阵重新变换，得到最终地图坐标 $(X_M, Y_M)$。

**坐标裁剪**：
$$
X_M = \max(0, \min(X_M, \text{WIDTH})),\quad Y_M = \max(0, \min(Y_M, \text{HEIGHT}))
$$

In [ ]:
import numpy as np
import cv2
from typing import Tuple

class CoordinateTransformer:
    """图像坐标 → 地图坐标（分层透视变换，无警告版）"""
    
    def __init__(self, M_ground, M_height_r, M_height_g, mask_image, map_w, map_h):
        self.M_ground = M_ground
        self.M_height_r = M_height_r
        self.M_height_g = M_height_g
        self.mask = mask_image
        self.W, self.H = map_w, map_h

    def _perspective(self, pt: Tuple[float, float], M: np.ndarray) -> Tuple[float, float]:
        src = np.array([[[pt[0], pt[1]]]], dtype=np.float32)
        dst = cv2.perspectiveTransform(src, M)
        x = float(np.clip(dst[0,0,0], 0, self.W))
        y = float(np.clip(dst[0,0,1], 0, self.H))
        return x, y

    def _get_layer(self, x: float, y: float) -> str:
        ix, iy = int(round(x)), int(round(y))
        ix = np.clip(ix, 0, self.mask.shape[1]-1)
        iy = np.clip(iy, 0, self.mask.shape[0]-1)
        b, g, r = self.mask[iy, ix]   # OpenCV BGR
        if b == g == r == 0:
            return "ground"
        if g > r and g > b:
            return "height_r"
        if b > r and b > g:
            return "height_g"
        return "default_r"

    def image_to_map(self, x_img: float, y_img: float) -> Tuple[float, float, str]:
        x_tmp, y_tmp = self._perspective((x_img, y_img), self.M_ground)
        layer = self._get_layer(x_tmp, y_tmp)
        if layer == "ground":
            xm, ym = x_tmp, y_tmp
        elif layer == "height_r":
            xm, ym = self._perspective((x_img, y_img), self.M_height_r)
        else:
            xm, ym = self._perspective((x_img, y_img), self.M_height_g)
        return xm, ym, layer

# 2. 卡尔曼滤波（位置平滑与预测）

## 数学原理

定义状态向量 $\mathbf{x} = [x, y, v_x, v_y]^T$，测量值 $\mathbf{z} = [x_{\text{meas}}, y_{\text{meas}}]^T$（来自识别框下边缘中心点）。

### 状态转移模型（恒定速度）

$$
\mathbf{x}_{k|k-1} = \mathbf{F} \mathbf{x}_{k-1|k-1},\quad
\mathbf{F} = \begin{bmatrix}
1 & 0 & \Delta t & 0 \\
0 & 1 & 0 & \Delta t \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

### 测量模型

$$
\mathbf{z}_k = \mathbf{H} \mathbf{x}_k + \mathbf{v}_k,\quad
\mathbf{H} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix},\quad
\mathbf{v}_k \sim \mathcal{N}(0, \mathbf{R})
$$

### 过程噪声协方差（恒定速度模型）

设 $q = \sigma_{\text{process}}^2$，则：

$$
\mathbf{Q} = q \cdot \begin{bmatrix}
\frac{\Delta t^4}{4} & 0 & \frac{\Delta t^3}{2} & 0 \\
0 & \frac{\Delta t^4}{4} & 0 & \frac{\Delta t^3}{2} \\
\frac{\Delta t^3}{2} & 0 & \Delta t^2 & 0 \\
0 & \frac{\Delta t^3}{2} & 0 & \Delta t^2
\end{bmatrix}
$$

### 卡尔曼滤波递推公式

**预测**：
$$
\begin{aligned}
\mathbf{x}_{k|k-1} &= \mathbf{F} \mathbf{x}_{k-1|k-1} \\
\mathbf{P}_{k|k-1} &= \mathbf{F} \mathbf{P}_{k-1|k-1} \mathbf{F}^T + \mathbf{Q}
\end{aligned}
$$

**更新**：
$$
\begin{aligned}
\mathbf{K}_k &= \mathbf{P}_{k|k-1} \mathbf{H}^T (\mathbf{H} \mathbf{P}_{k|k-1} \mathbf{H}^T + \mathbf{R})^{-1} \\
\mathbf{x}_{k|k} &= \mathbf{x}_{k|k-1} + \mathbf{K}_k (\mathbf{z}_k - \mathbf{H} \mathbf{x}_{k|k-1}) \\
\mathbf{P}_{k|k} &= (\mathbf{I} - \mathbf{K}_k \mathbf{H}) \mathbf{P}_{k|k-1}
\end{aligned}
$$

In [ ]:
import time
import numpy as np
from typing import Tuple

class KalmanFilter2D:
    """2D 位置-速度卡尔曼滤波器（无警告版）"""
    
    def __init__(self, process_noise_std: float = 0.1, meas_noise_std: float = 1.0):
        self.state = np.zeros(4)          # [x, y, vx, vy]
        self.P = np.eye(4) * 100.0
        self.H = np.array([[1,0,0,0],[0,1,0,0]])
        self.R = np.eye(2) * (meas_noise_std**2)
        self.last_time = time.time()
        self.q = process_noise_std**2

    def predict(self) -> Tuple[float, float]:
        now = time.time()
        dt = max(0.001, now - self.last_time)
        # 状态转移矩阵 F
        F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]])
        # 过程噪声 Q
        dt2, dt3, dt4 = dt**2, dt**3, dt**4
        Q = self.q * np.array([
            [dt4/4, 0, dt3/2, 0],
            [0, dt4/4, 0, dt3/2],
            [dt3/2, 0, dt2, 0],
            [0, dt3/2, 0, dt2]
        ])
        self.state = F @ self.state
        self.P = F @ self.P @ F.T + Q
        self.last_time = now
        return self.state[0], self.state[1]

    def update(self, x: float, y: float):
        z = np.array([x, y])
        S = self.H @ self.P @ self.H.T + self.R
        try:
            K = self.P @ self.H.T @ np.linalg.inv(S)
        except np.linalg.LinAlgError:
            K = self.P @ self.H.T @ np.linalg.pinv(S)
        y_vec = z - self.H @ self.state
        self.state = self.state + K @ y_vec
        self.P = (np.eye(4) - K @ self.H) @ self.P

    def get_position(self) -> Tuple[float, float]:
        return self.state[0], self.state[1]

    def get_velocity(self) -> Tuple[float, float]:
        return self.state[2], self.state[3]

# 3. 数据关联（IoU + 贪婪匹配）

## 数学原理

**IoU（交并比）** 用于衡量检测框与跟踪器预测框的相似度：

$$
\text{IoU}(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

其中 $A = (x_A,y_A,w_A,h_A)$，$B = (x_B,y_B,w_B,h_B)$，

$$
\begin{aligned}
\text{inter\_x1} &= \max(x_A, x_B) \\
\text{inter\_y1} &= \max(y_A, y_B) \\
\text{inter\_x2} &= \min(x_A+w_A, x_B+w_B) \\
\text{inter\_y2} &= \min(y_A+h_A, y_B+h_B) \\
\text{inter\_area} &= \max(0, \text{inter\_x2} - \text{inter\_x1}) \times \max(0, \text{inter\_y2} - \text{inter\_y1}) \\
\text{union\_area} &= w_A h_A + w_B h_B - \text{inter\_area} \\
\text{IoU} &= \frac{\text{inter\_area}}{\text{union\_area}}
\end{aligned}
$$

**匹配策略**（贪婪简化版匈牙利算法）：
1. 计算所有检测与所有预测框的 IoU。
2. 筛选 IoU ≥ 阈值（如 0.3）的候选对。
3. 按 IoU 降序排序，依次将未匹配的检测与未匹配的跟踪器配对。

In [ ]:
from typing import List, Tuple

class HungarianMatcher:
    @staticmethod
    def compute_iou(box1: Tuple[float, float, float, float],
                    box2: Tuple[float, float, float, float]) -> float:
        x1,y1,w1,h1 = box1
        x2,y2,w2,h2 = box2
        inter_x1 = max(x1, x2)
        inter_y1 = max(y1, y2)
        inter_x2 = min(x1+w1, x2+w2)
        inter_y2 = min(y1+h1, y2+h2)
        if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
            return 0.0
        inter = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
        union = w1*h1 + w2*h2 - inter
        return inter / union if union > 0 else 0.0

    @staticmethod
    def match(detections: List[Tuple], trackers: List, iou_thresh: float = 0.3) -> List[Tuple[int, int]]:
        """
        detections: [(robot_id, (x,y,w,h), conf), ...]
        trackers:   每个 tracker 有 get_predicted_box() 方法
        返回: [(det_idx, tracker_idx), ...]
        """
        if not detections or not trackers:
            return []
        # 收集有效的预测框
        pred_boxes = []
        valid_idx = []
        for i, trk in enumerate(trackers):
            box = trk.get_predicted_box()
            if box is not None:
                pred_boxes.append(box)
                valid_idx.append(i)
        if not pred_boxes:
            return []
        # 构建候选匹配
        candidates = []
        for i, (_, det_box, _) in enumerate(detections):
            for j, pred_box in enumerate(pred_boxes):
                iou = HungarianMatcher.compute_iou(det_box, pred_box)
                if iou >= iou_thresh:
                    candidates.append((iou, i, j))
        candidates.sort(reverse=True, key=lambda x: x[0])
        used_det = [False] * len(detections)
        used_trk = [False] * len(pred_boxes)
        matches = []
        for iou, i, j in candidates:
            if not used_det[i] and not used_trk[j]:
                matches.append((i, valid_idx[j]))
                used_det[i] = used_trk[j] = True
        return matches

# 4. 机器人跟踪器与跟踪管理器

## 跟踪器状态机

- **新建**：首次检测到机器人时创建，初始位置为检测框下边缘中心点。
- **更新**：匹配成功后用测量值更新卡尔曼滤波器，重置丢失计数。
- **丢失**：连续 $N_{\text{miss}}$（默认 10）次未匹配 → 标记为不活跃。
- **过期**：最后更新时间超过 3 秒 → 从管理器中移除。

## 预测框生成

根据估计速度 $v = \sqrt{v_x^2+v_y^2}$ 动态调整预测框大小：

$$
\text{size} = \text{clip}(30 + 5v,\ 20,\ 100)
$$

预测框中心为卡尔曼预测位置 $(x_{\text{pred}}, y_{\text{pred}})$。

## 完整工作流程

1. 对每帧检测到的机器人，使用 `CoordinateTransformer` 将图像下边缘中心点转换为地图坐标。
2. 将地图坐标作为测量值传入 `TrackingManager.update()`。
3. 管理器执行：预测 → 匹配 → 更新/新建/标记丢失 → 移除过期。
4. 输出所有活跃跟踪器的平滑位置 $(x,y)$。

In [ ]:
import time
import numpy as np
from typing import Dict, List, Tuple, Optional

class RobotTracker:
    def __init__(self, robot_id: str, x: float, y: float):
        self.robot_id = robot_id
        self.kf = KalmanFilter2D()
        self.kf.state = np.array([x, y, 0.0, 0.0])
        self.last_update = time.time()
        self.miss_count = 0
        self.active = True
        self.predicted_box = None

    def update(self, x: float, y: float):
        self.kf.update(x, y)
        self.last_update = time.time()
        self.miss_count = 0
        self.predicted_box = None   # 下次 predict 时重新生成

    def predict(self) -> Tuple[float, float]:
        px, py = self.kf.predict()
        vx, vy = self.kf.get_velocity()
        speed = np.hypot(vx, vy)
        size = np.clip(30 + speed * 5, 20, 100)
        self.predicted_box = (px - size/2, py - size/2, size, size)
        return px, py

    def get_predicted_box(self) -> Optional[Tuple[float, float, float, float]]:
        return self.predicted_box

    def get_position(self) -> Tuple[float, float]:
        return self.kf.get_position()

    def mark_miss(self):
        self.miss_count += 1
        if self.miss_count > 10:
            self.active = False

    def is_stale(self, timeout: float = 3.0) -> bool:
        return (time.time() - self.last_update) > timeout


class TrackingManager:
    def __init__(self):
        self.trackers: Dict[str, RobotTracker] = {}

    def update(self, detections: List[Tuple[str, Tuple[float, float, float, float], float]]) -> Dict[str, Tuple[float, float]]:
        """
        detections: [(robot_id, (x_map, y_map, w, h), conf), ...]
        注意: (x_map, y_map) 已经是地图坐标的下边缘中心点
        """
        # 1. 预测所有现有跟踪器
        for trk in self.trackers.values():
            if trk.active:
                trk.predict()

        # 2. 匹配
        tracker_list = list(self.trackers.values())
        matches = HungarianMatcher.match(detections, tracker_list)

        matched_det = set()
        matched_trk = set()

        for det_idx, trk_idx in matches:
            det_id, (x, y, w, h), _ = detections[det_idx]
            trk = tracker_list[trk_idx]
            if det_id == trk.robot_id:
                trk.update(x, y)
                matched_det.add(det_idx)
                matched_trk.add(trk_idx)

        # 3. 未匹配的检测 → 新建跟踪器
        for i, (det_id, (x, y, w, h), _) in enumerate(detections):
            if i not in matched_det:
                self.trackers[det_id] = RobotTracker(det_id, x, y)

        # 4. 未匹配的跟踪器 → 标记丢失
        for i, trk in enumerate(tracker_list):
            if i not in matched_trk:
                trk.mark_miss()

        # 5. 移除过期/非活跃跟踪器
        to_del = [rid for rid, trk in self.trackers.items() if trk.is_stale() or not trk.active]
        for rid in to_del:
            del self.trackers[rid]

        # 返回所有活跃位置
        return {rid: trk.get_position() for rid, trk in self.trackers.items() if trk.active}

# 5. 使用示例

以下代码演示如何将上述模块串联起来，完成从图像坐标到跟踪位置的完整流程。

In [ ]:
# 示例：模拟运行（请替换为实际标定数据）
if __name__ == "__main__":
    # 模拟标定数据（实际应读取 .npy 文件）
    M_ground = np.eye(3, dtype=np.float32)
    M_height_r = np.eye(3, dtype=np.float32)
    M_height_g = np.eye(3, dtype=np.float32)
    mask = np.zeros((1080, 1920, 3), dtype=np.uint8)   # 实际应加载掩码图
    transformer = CoordinateTransformer(M_ground, M_height_r, M_height_g, mask, 1920, 1080)
    tracker_mgr = TrackingManager()

    # 模拟一帧检测结果（图像坐标下的下边缘中心点）
    detections_img = [("R1", (640, 480, 100, 150), 0.9)]   # (id, (x,y,w,h), conf)
    # 转换为地图坐标
    detections_map = []
    for robot_id, (x, y, w, h), conf in detections_img:
        xm, ym, layer = transformer.image_to_map(x, y)   # 传入下边缘中心点
        detections_map.append((robot_id, (xm, ym, w, h), conf))

    # 更新跟踪器
    positions = tracker_mgr.update(detections_map)
    print("当前跟踪位置:", positions)